# Modelling the Limit Order Book

## Simulation of LOBs

### Probabilistic generative models

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt


class LimitOrderBook:
    """Price-time priority LOB matching engine."""

    def __init__(self, tick_size=0.01):
        self.tick_size = tick_size
        self.bids = {}   # price -> volume
        self.asks = {}   # price -> volume
        self.best_bid = None
        self.best_ask = None
        self.mid_price = None

    def _snap(self, p):
        return round(round(p / self.tick_size) * self.tick_size, 8)

    def _refresh(self):
        self.bids = {p: v for p, v in self.bids.items() if v > 1e-9}
        self.asks = {p: v for p, v in self.asks.items() if v > 1e-9}
        self.best_bid = max(self.bids) if self.bids else None
        self.best_ask = min(self.asks) if self.asks else None
        if self.best_bid is not None and self.best_ask is not None:
            self.mid_price = (self.best_bid + self.best_ask) / 2.0

    def add_limit(self, side, price, volume):
        price = self._snap(price)
        if side == 'buy':
            while volume > 1e-9 and self.asks and min(self.asks) <= price:
                best = min(self.asks)
                trade = min(self.asks[best], volume)
                self.asks[best] -= trade
                volume -= trade
                self._refresh()
            if volume > 1e-9:
                self.bids[price] = self.bids.get(price, 0) + volume
        else:
            while volume > 1e-9 and self.bids and max(self.bids) >= price:
                best = max(self.bids)
                trade = min(self.bids[best], volume)
                self.bids[best] -= trade
                volume -= trade
                self._refresh()
            if volume > 1e-9:
                self.asks[price] = self.asks.get(price, 0) + volume
        self._refresh()

    def add_market(self, side, volume):
        filled = 0
        if side == 'buy':
            while volume > 1e-9 and self.asks:
                best = min(self.asks)
                trade = min(self.asks[best], volume)
                self.asks[best] -= trade
                volume -= trade
                filled += trade
        else:
            while volume > 1e-9 and self.bids:
                best = max(self.bids)
                trade = min(self.bids[best], volume)
                self.bids[best] -= trade
                volume -= trade
                filled += trade
        self._refresh()
        return filled

    def cancel(self, side, price, vol=None):
        price = self._snap(price)
        book = self.bids if side == 'buy' else self.asks
        if price in book:
            if vol is None or vol >= book[price]:
                del book[price]
            else:
                book[price] = max(0.0, book[price] - vol)
        self._refresh()

    def snapshot(self, n=12):
        bids = sorted([(p, v) for p, v in self.bids.items()], reverse=True)[:n]
        asks = sorted([(p, v) for p, v in self.asks.items()])[:n]
        return bids, asks


def plot_lob_dynamics(snapshots, title='LOB Dynamics'):
    times = [s['t'] for s in snapshots]
    mids  = [s['mid'] for s in snapshots]

    t_b, p_b, v_b = [], [], []
    t_a, p_a, v_a = [], [], []
    for s in snapshots:
        for price, vol in s['bids']:
            t_b.append(s['t']); p_b.append(price); v_b.append(vol)
        for price, vol in s['asks']:
            t_a.append(s['t']); p_a.append(price); v_a.append(vol)

    t_b = np.array(t_b); p_b = np.array(p_b); v_b = np.array(v_b)
    t_a = np.array(t_a); p_a = np.array(p_a); v_a = np.array(v_a)

    all_v = np.concatenate([v_b, v_a]) if (len(v_b) and len(v_a)) else np.array([1.0])
    vref  = max(np.percentile(all_v, 90), 1e-9)
    def sz(v): return np.clip(v / vref, 0.05, 1.5) * 50 + 5

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(12, 8),
        gridspec_kw={'height_ratios': [1, 3]}, sharex=True
    )
    fig.subplots_adjust(hspace=0.06)

    ax1.plot(times, mids, color='black', linewidth=1.2)
    ax1.set_ylabel('Mid-price')
    ax1.set_title(title, fontsize=11)

    if len(t_b):
        ax2.scatter(t_b, p_b, s=sz(v_b), c='#2ca02c', alpha=0.35,
                    linewidths=0, label='Bid limit orders')
    if len(t_a):
        ax2.scatter(t_a, p_a, s=sz(v_a), c='#d62728', alpha=0.35,
                    linewidths=0, label='Ask limit orders')
    ax2.plot(times, mids, color='black', linewidth=1.0,
             linestyle='--', alpha=0.7, label='Mid-price')
    ax2.set_xlabel('Time (periods)')
    ax2.set_ylabel('Price')
    ax2.legend(loc='upper right', fontsize=9)
    plt.show()


Simulation with Poisson order arrivals, log-normal order sizes, and geometric price placement relative to the best quote. The matching engine runs price-time priority continuously.

In [2]:
rng  = np.random.default_rng(42)
lob  = LimitOrderBook(tick_size=0.5)
P0, tick = 100.0, 0.5

for i in range(10):
    lob.bids[round(P0 - (i + 0.5) * tick, 8)] = max(1, int(rng.lognormal(2.2, 0.7)))
    lob.asks[round(P0 + (i + 0.5) * tick, 8)] = max(1, int(rng.lognormal(2.2, 0.7)))
lob._refresh()

snapshots_prob = []

for t in range(3000):
    if lob.best_bid is None or lob.best_ask is None:
        break
    for _ in range(int(rng.poisson(2))):
        if lob.best_bid is None or lob.best_ask is None:
            break
        side = 'buy' if rng.random() < 0.5 else 'sell'
        sz   = max(1, int(rng.lognormal(2.2, 0.7)))
        if rng.random() < 0.25:
            opp = sum(lob.asks.values()) if side == 'buy' else sum(lob.bids.values())
            lob.add_market(side, min(sz, max(1, int(opp * 0.4))))
        else:
            n_ticks = min(int(rng.geometric(0.45)) - 1, 20)
            price = (lob.best_bid - n_ticks * tick) if side == 'buy' \
                    else (lob.best_ask + n_ticks * tick)
            lob.add_limit(side, price, sz)
    if t % 8 == 7:
        if len(lob.bids) > 20:
            lob.cancel('buy', min(lob.bids))
        if len(lob.asks) > 20:
            lob.cancel('sell', max(lob.asks))
    if t % 15 == 0 and lob.mid_price is not None:
        bids, asks = lob.snapshot(12)
        snapshots_prob.append({'t': t, 'bids': bids, 'asks': asks, 'mid': lob.mid_price})

plot_lob_dynamics(
    snapshots_prob,
    title='Probabilistic Generative Model – LOB Dynamics\n'
          '(Poisson arrivals, log-normal sizes, geometric price placement)'
)
print(f'{len(snapshots_prob)} snapshots, '
      f'mid-price: {snapshots_prob[0]["mid"]:.2f} → {snapshots_prob[-1]["mid"]:.2f}')


KeyboardInterrupt: 

### Agent-based models

McGroarty et al. (2019) five-agent simulation: market makers, liquidity consumers, momentum traders, mean reversion traders, and noise traders.

In [ ]:
rng  = np.random.default_rng(0)
lob  = LimitOrderBook(tick_size=0.5)
P0, tick = 100.0, 0.5
T, snap_every = 10000, 50

for i in range(10):
    lob.bids[round(P0 - (i + 0.5) * tick, 8)] = max(1, int(rng.lognormal(2.0, 0.5)))
    lob.asks[round(P0 + (i + 0.5) * tick, 8)] = max(1, int(rng.lognormal(2.0, 0.5)))
lob._refresh()

# ── agent counts and action probabilities (McGroarty et al. Table 1) ──────────
n_mm, n_lc, n_mr, n_mt, n_nt = 2, 2, 5, 5, 12
d_mm, d_lc, d_mr, d_mt, d_nt = 0.10, 0.10, 0.40, 0.40, 0.75

# market maker state
mm_w = 50
mm_bid_p = [None]*n_mm; mm_bid_v = [0]*n_mm
mm_ask_p = [None]*n_mm; mm_ask_v = [0]*n_mm
order_signs = []

# liquidity consumer state
lc_side = ['buy' if rng.random() < 0.5 else 'sell' for _ in range(n_lc)]
lc_rem  = [int(rng.uniform(300, 1200)) for _ in range(n_lc)]

# momentum trader state
mt_nr, mt_kappa, mt_scale = 200, 0.001, 4000
price_hist = [P0] * mt_nr

# mean reversion trader state
mr_alpha, mr_k, mr_vmr = 0.02, 1.5, 8
mr_ema = [P0]*n_mr
mr_var = [0.0]*n_mr

# noise trader state
nt_sz_mu, nt_sz_sig = 1.8, 0.65
nt_lam_m, nt_lam_l  = 0.25, 0.60          # lam_c = 0.15
nt_lcrs, nt_linspr, nt_lspr = 0.15, 0.20, 0.40   # loffspr = 0.25
nt_beta, nt_xmin = 3.0, 0.5
nt_orders = [[] for _ in range(n_nt)]

snapshots_abm = []

for t in range(T):
    if lob.best_bid is None or lob.best_ask is None:
        break

    mid   = lob.mid_price
    t_sgn = 0

    # ── market makers ────────────────────────────────────────────────────────
    for i in range(n_mm):
        if rng.random() >= d_mm:
            continue
        if lob.best_bid is None or lob.best_ask is None:
            continue

        pred_buy = (np.mean(order_signs[-mm_w:]) > 0) if len(order_signs) >= mm_w \
                   else (rng.random() < 0.5)

        if mm_bid_p[i] is not None and mm_bid_p[i] in lob.bids:
            lob.cancel('buy', mm_bid_p[i], mm_bid_v[i])
        if mm_ask_p[i] is not None and mm_ask_p[i] in lob.asks:
            lob.cancel('sell', mm_ask_p[i], mm_ask_v[i])

        if lob.best_bid is None or lob.best_ask is None:
            continue

        if pred_buy:
            sell_v = max(1, int(rng.uniform(5, 25))); buy_v = 2
        else:
            buy_v  = max(1, int(rng.uniform(5, 25))); sell_v = 2

        bp = lob.best_bid; ap = lob.best_ask
        lob.bids[bp] = lob.bids.get(bp, 0) + buy_v
        lob.asks[ap] = lob.asks.get(ap, 0) + sell_v
        mm_bid_p[i], mm_bid_v[i] = bp, buy_v
        mm_ask_p[i], mm_ask_v[i] = ap, sell_v
    lob._refresh()

    # ── liquidity consumers ──────────────────────────────────────────────────
    for i in range(n_lc):
        if lc_rem[i] <= 0 or rng.random() >= d_lc:
            continue
        if lob.best_bid is None or lob.best_ask is None:
            continue
        side = lc_side[i]
        opp  = lob.asks if side == 'buy' else lob.bids
        if not opp:
            continue
        best_opp = min(opp) if side == 'buy' else max(opp)
        vol = min(lc_rem[i], opp[best_opp])
        if vol > 0:
            lob.add_market(side, vol)
            lc_rem[i] -= vol
            t_sgn = 1 if side == 'buy' else -1

    # ── momentum traders ─────────────────────────────────────────────────────
    price_hist.append(mid)
    if len(price_hist) > mt_nr + 100:
        price_hist.pop(0)

    for i in range(n_mt):
        if rng.random() >= d_mt or len(price_hist) < mt_nr + 1:
            continue
        roc  = (price_hist[-1] - price_hist[-mt_nr - 1]) / price_hist[-mt_nr - 1]
        if abs(roc) < mt_kappa:
            continue
        side = 'buy' if roc > 0 else 'sell'
        vol  = max(1, int(abs(roc) * mt_scale))
        opp  = sum(lob.asks.values()) if side == 'buy' else sum(lob.bids.values())
        vol  = min(vol, max(1, int(opp * 0.25)))
        lob.add_market(side, vol)
        t_sgn = 1 if side == 'buy' else -1

    # ── mean reversion traders ───────────────────────────────────────────────
    for i in range(n_mr):
        if rng.random() >= d_mr:
            continue
        if lob.best_bid is None or lob.best_ask is None:
            continue
        mr_ema[i] += mr_alpha * (mid - mr_ema[i])
        dev = mid - mr_ema[i]
        mr_var[i] += mr_alpha * (dev * dev - mr_var[i])
        sigma = max(np.sqrt(max(mr_var[i], 0.0)), tick * 0.5)

        if dev >= mr_k * sigma:
            p = lob.best_ask - tick
            if p > lob.best_bid:
                lob.add_limit('sell', p, mr_vmr)
            else:
                lob.add_limit('sell', lob.best_ask, mr_vmr)
        elif dev <= -mr_k * sigma:
            p = lob.best_bid + tick
            if p < lob.best_ask:
                lob.add_limit('buy', p, mr_vmr)
            else:
                lob.add_limit('buy', lob.best_bid, mr_vmr)

    # ── noise traders ────────────────────────────────────────────────────────
    for i in range(n_nt):
        if rng.random() >= d_nt:
            continue
        if lob.best_bid is None or lob.best_ask is None:
            continue

        side = 'buy' if rng.random() < 0.5 else 'sell'
        sz   = max(1, int(rng.lognormal(nt_sz_mu, nt_sz_sig)))
        r    = rng.random()

        if r < nt_lam_m:
            opp = sum(lob.asks.values()) if side == 'buy' else sum(lob.bids.values())
            lob.add_market(side, min(sz, max(1, int(opp * 0.5))))
            t_sgn = 1 if side == 'buy' else -1
        elif r < nt_lam_m + nt_lam_l:
            rl = rng.random()
            if rl < nt_lcrs:
                price = lob.best_ask if side == 'buy' else lob.best_bid
            elif rl < nt_lcrs + nt_linspr:
                price = round(rng.uniform(lob.best_bid, lob.best_ask) / tick) * tick
            elif rl < nt_lcrs + nt_linspr + nt_lspr:
                price = lob.best_bid if side == 'buy' else lob.best_ask
            else:
                u      = max(rng.random(), 1e-9)
                offset = nt_xmin * (1.0 - u) ** (-1.0 / (nt_beta - 1))
                offset = min(offset, 20 * tick)
                price  = (lob.best_bid - offset) if side == 'buy' \
                         else (lob.best_ask + offset)
                price  = round(price / tick) * tick
            lob.add_limit(side, price, sz)
            nt_orders[i].append((side, price))
            t_sgn = 1 if side == 'buy' else -1
        else:
            if nt_orders[i]:
                s, p = nt_orders[i].pop(0)
                lob.cancel(s, p)

        if not lob.bids and lob.best_ask is not None:
            lob.bids[lob.best_ask - tick] = 5; lob._refresh()
        if not lob.asks and lob.best_bid is not None:
            lob.asks[lob.best_bid + tick] = 5; lob._refresh()

    order_signs.append(t_sgn)

    if t % snap_every == 0 and lob.mid_price is not None:
        bids, asks = lob.snapshot(12)
        snapshots_abm.append({'t': t, 'bids': bids, 'asks': asks, 'mid': lob.mid_price})

plot_lob_dynamics(
    snapshots_abm,
    title='Agent-Based Model (McGroarty et al. 2019) – LOB Dynamics\n'
          '(Market makers · Liquidity consumers · Momentum · Mean reversion · Noise traders)'
)
print(f'{len(snapshots_abm)} snapshots, '
      f'mid-price: {snapshots_abm[0]["mid"]:.2f} → {snapshots_abm[-1]["mid"]:.2f}')
